# Phase 2: Stratified Dataset Splitting

## Objective

Before training and evaluating a deep learning model,
the dataset must be divided into separate subsets.

We create three splits:

1. Training Set
2. Validation Set
3. Test Set

### Why split the dataset?

Each subset serves a different purpose:

| Split | Purpose |
|---------|---------|
| Training | Learn model parameters |
| Validation | Tune model and monitor performance |
| Test | Final unbiased evaluation |

### Split Ratios

For this project we use:

- Training = 70%
- Validation = 15%
- Test = 15%

The splitting process is performed separately for each class:

- leaf
- not_leaf

This preserves class balance across all subsets.

A fixed random seed is used to ensure reproducibility.


# Import Required Libraries

We import:

- os → directory operations
- shutil → file copying
- random → random shuffling
- pathlib → path management

These libraries help organize images into
training, validation, and test folders.

In [ ]:
#Import Libraries

import os
import shutil
import random
from pathlib import Path
from google.colab import drive

# Reproducibility Setup

Dataset splitting involves random shuffling.

To ensure the same images are assigned to the same subsets
across multiple runs, we set a fixed random seed.

Benefits:

- consistent experiments
- repeatable results
- easier debugging
- research reproducibility

In [ ]:
SEED = 42

random.seed(SEED)

print(f"✅ Random seed set to {SEED}")

✅ Random seed set to 42


# Mounting Google Drive (Robust Connection)

Sometimes Google Drive gets "stuck" in Colab, causing the error:
`ValueError: Mountpoint must not already contain files`.

To ensure a clean and stable connection, we:
1. Forcefully unmount any existing Drive connection in the background.
2. Completely delete the corrupted `/content/drive` directory.
3. Mount Google Drive completely fresh.

In [ ]:
# 1. Forcefully unmount the drive in the background
!fusermount -u /content/drive 2>/dev/null || true

# 2. Completely delete the /content/drive directory if it exists
if os.path.exists('/content/drive'):
    shutil.rmtree('/content/drive')
    print("✅ Deleted corrupted /content/drive folder")

# 3. Mount it completely fresh
drive.mount('/content/drive')

✅ Deleted corrupted /content/drive folder
Mounted at /content/drive


In [ ]:

print("🧹 Cleaning up old splits...")

# Delete old train/val/test folders if they exist
for folder in ["train", "val", "test"]:
    path = os.path.join("/content/drive/MyDrive/cardamom_dataset", folder)
    if os.path.exists(path):
        shutil.rmtree(path)
        print(f"✅ Deleted {folder}/")
    else:
        print(f"⚠️ {folder}/ doesn't exist (first run)")

print("✅ Cleanup complete - ready for fresh splits!")

🧹 Cleaning up old splits...
✅ Deleted train/
✅ Deleted val/
✅ Deleted test/
✅ Cleanup complete - ready for fresh splits!


# Dataset Configuration

The dataset is stored inside Google Drive.

Original images are located in:

cardamom_dataset/raw/

The following directories will be created:

- train/
- val/
- test/

In [ ]:
RAW_DIR = "/content/drive/MyDrive/cardamom_dataset/raw"

TRAIN_DIR = "/content/drive/MyDrive/cardamom_dataset/train"
VAL_DIR = "/content/drive/MyDrive/cardamom_dataset/val"
TEST_DIR = "/content/drive/MyDrive/cardamom_dataset/test"

# Creating Split Directories

Each dataset subset contains:

- leaf/
- not_leaf/

This directory structure is compatible with
PyTorch ImageFolder.

In [ ]:
for directory in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    for class_name in ["leaf", "not_leaf"]:
        os.makedirs(
            os.path.join(directory, class_name),
            exist_ok=True
        )

print("✅ Dataset directories created")

✅ Dataset directories created


# Stratified Splitting Function

The dataset is split separately for each class.

This process:

1. Reads all image filenames
2. Randomly shuffles image order
3. Calculates split boundaries
4. Creates:
   - training subset
   - validation subset
   - test subset
5. Copies images into their respective folders

By splitting each class independently, we preserve
class balance across all dataset subsets.


In [ ]:
def split_class(
    class_name,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15
):

    # Source directory
    src_dir = os.path.join(
        RAW_DIR,
        class_name
    )

    # Read image filenames
    files = [
        f
        for f in os.listdir(src_dir)
        if f.lower().endswith(
            (
                ".jpg",
                ".jpeg",
                ".png"
            )
        )
    ]

    # Randomize image order
    random.shuffle(files)

    # Total number of images
    n = len(files)

    # Calculate split indices
    train_end = int(
        n * train_ratio
    )

    val_end = train_end + int(
        n * val_ratio
    )

    # Create subsets
    train_files = files[:train_end]

    val_files = files[
        train_end:val_end
    ]

    test_files = files[
        val_end:
    ]

    # Copy training images
    for filename in train_files:

        shutil.copy2(
            os.path.join(
                src_dir,
                filename
            ),
            os.path.join(
                TRAIN_DIR,
                class_name,
                filename
            )
        )

    # Copy validation images
    for filename in val_files:

        shutil.copy2(
            os.path.join(
                src_dir,
                filename
            ),
            os.path.join(
                VAL_DIR,
                class_name,
                filename
            )
        )

    # Copy test images
    for filename in test_files:

        shutil.copy2(
            os.path.join(
                src_dir,
                filename
            ),
            os.path.join(
                TEST_DIR,
                class_name,
                filename
            )
        )

    # Display summary
    print(
        f"✅ {class_name}: "
        f"{len(train_files)} train | "
        f"{len(val_files)} val | "
        f"{len(test_files)} test"
    )

# Execute Dataset Splitting

The splitting function is now applied to both classes:

- leaf
- not_leaf

Each class is processed independently to ensure
that class proportions remain balanced.

In [ ]:
print("=" * 60)
print("📊 SPLITTING DATASET")
print("=" * 60)

split_class("leaf")

split_class("not_leaf")

print("\n🎉 Dataset splitting completed!")

📊 SPLITTING DATASET
✅ leaf: 91 train | 19 val | 20 test
✅ not_leaf: 91 train | 19 val | 20 test

🎉 Dataset splitting completed!


# Verify Dataset Statistics

After splitting, it is important to verify that
the expected number of images exists inside each
dataset subset.

This helps confirm that the splitting process
was performed correctly.

In [ ]:
for split_name, split_dir in [

    ("Train", TRAIN_DIR),
    ("Validation", VAL_DIR),
    ("Test", TEST_DIR)

]:

    print(f"\n📁 {split_name} Set")

    total_images = 0

    for class_name in [

        "leaf",
        "not_leaf"

    ]:

        class_dir = os.path.join(
            split_dir,
            class_name
        )

        count = len(
            os.listdir(class_dir)
        )

        total_images += count

        print(
            f"   {class_name}: {count}"
        )

    print(
        f"   Total: {total_images}"
    )


📁 Train Set
   leaf: 91
   not_leaf: 91
   Total: 182

📁 Validation Set
   leaf: 19
   not_leaf: 19
   Total: 38

📁 Test Set
   leaf: 20
   not_leaf: 20
   Total: 40


# Dataset Structure Overview

After splitting, the dataset structure becomes:

cardamom_dataset
├── train
│   ├── leaf
│   └── not_leaf
├── val
│   ├── leaf
│   └── not_leaf
└── test
    ├── leaf
    └── not_leaf

This structure is fully compatible with
PyTorch ImageFolder and DataLoader.


In [ ]:
print("\n📁 Dataset Directories")

print(f"\nTraining Set:")
print(TRAIN_DIR)

print(f"\nValidation Set:")
print(VAL_DIR)

print(f"\nTest Set:")
print(TEST_DIR)


📁 Dataset Directories

Training Set:
/content/drive/MyDrive/cardamom_dataset/train

Validation Set:
/content/drive/MyDrive/cardamom_dataset/val

Test Set:
/content/drive/MyDrive/cardamom_dataset/test
